# Content Moderation System — Colab Training (Phase 1 + Phase 2)

Train both phases on a **free Colab GPU** with zero load on your Mac.

| Profile | Sample size | Epochs | Est. time (T4) |
|---|---|---|---|
| `quick` | 2,000 | 1 | ~25–35 min |
| `standard` | 5,000 | 2 | ~45–70 min |
| `full` | 159K (full Jigsaw) | 3 | ~2–3 hrs |

**Before running:** Runtime → Change runtime type → **T4 GPU**

**Outputs:** trained checkpoints, calibration plots, evaluation report, zip download.

## 1. Configuration

In [ ]:
# @title User settings
REPO_URL = "https://github.com/hschinmayabharadwaj/Content-Moderation.git"  # @param {type:"string"}
BRANCH = "main"  # @param {type:"string"}
PROFILE = "standard"  # @param ["quick", "standard", "full"]

PROFILES = {
    "quick": {"sample_size": 2000, "epochs": 1, "phase2_sample": 2000},
    "standard": {"sample_size": 5000, "epochs": 2, "phase2_sample": 5000},
    "full": {"sample_size": None, "epochs": 3, "phase2_sample": 10000},
}

cfg = PROFILES[PROFILE]
SAMPLE_SIZE = cfg["sample_size"]
NUM_EPOCHS = cfg["epochs"]
PHASE2_SAMPLE = cfg["phase2_sample"]

print(f"Profile: {PROFILE}")
print(f"Phase 1 sample: {SAMPLE_SIZE or 'full dataset'}")
print(f"Epochs: {NUM_EPOCHS}")
print(f"Phase 2 combined sample: {PHASE2_SAMPLE}")

## 2. GPU check & clone repository

In [ ]:
import os
import subprocess
import sys
import zipfile
import urllib.request

import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU detected. Go to Runtime → Change runtime type → T4 GPU, then re-run."
    )

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"CUDA: {torch.version.cuda}")

REPO_DIR = "/content/content-moderation-system"

if os.path.exists(REPO_DIR):
    subprocess.run(["rm", "-rf", REPO_DIR], check=True)

clone = subprocess.run(
    ["git", "clone", "--branch", BRANCH, "--depth", "1", REPO_URL, REPO_DIR],
    capture_output=True,
    text=True,
)

if clone.returncode != 0:
    print("git clone failed, falling back to GitHub zip download...")
    print(clone.stderr.strip())
    zip_url = REPO_URL.replace(".git", f"/archive/refs/heads/{BRANCH}.zip")
    zip_path = "/content/repo.zip"
    urllib.request.urlretrieve(zip_url, zip_path)
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall("/content")
    extracted = f"/content/Content-Moderation-{BRANCH}"
    os.rename(extracted, REPO_DIR)
    print(f"Downloaded and extracted to {REPO_DIR}")
else:
    print(f"Cloned to {REPO_DIR}")

os.chdir(REPO_DIR)
!ls -la

## 3. Install dependencies

In [ ]:
import subprocess
import sys

def pip_install(*packages):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])

# Fix Colab if a prior run upgraded numpy/pandas (pandas 3.x / numpy 2.5.x)
pip_install("pandas==2.2.2", "numpy>=1.22,<2.1")

pip_install(
    "transformers", "datasets", "scikit-learn", "pyyaml", "tqdm",
    "matplotlib", "seaborn", "scipy", "fasttext-wheel", "langdetect",
)

import numpy as np
import pandas as pd
print(f"numpy {np.__version__} | pandas {pd.__version__}")
print("Dependencies OK — conflict warnings above can be ignored if versions match Colab.")

## 4. Build Colab configs

In [ ]:
import copy
import yaml

PHASE1_CONFIG = {
    "model": {
        "name": "distilbert-base-uncased",
        "num_labels": 6,
        "dropout": 0.1,
        "max_length": 256,
        "freeze_embeddings": False,
        "freeze_encoder_layers": 0,
        "hidden_size": 256,
        "use_intermediate_layer": True,
    },
    "training": {
        "train_file": "data/train.csv",
        "validation_file": "data/validation.csv",
        "test_file": "data/test.csv",
        "use_sample": True,
        "batch_size": 32,
        "learning_rate": 3.0e-5,
        "num_epochs": NUM_EPOCHS,
        "warmup_steps": 100,
        "weight_decay": 0.01,
        "max_grad_norm": 1.0,
        "optimizer": "adamw",
        "scheduler": "linear",
        "use_class_weights": True,
        "focal_loss": False,
        "focal_alpha": 0.25,
        "focal_gamma": 2.0,
        "use_augmentation": False,
        "device": "cuda",
        "mixed_precision": True,
        "num_workers": 2,
        "pin_memory": True,
        "save_dir": "models",
        "save_strategy": "best",
        "save_total_limit": 2,
        "load_best_at_end": True,
        "eval_strategy": "epoch",
        "eval_steps": 500,
        "logging_steps": 50,
        "early_stopping": True,
        "early_stopping_patience": 2,
        "early_stopping_metric": "eval_loss",
    },
    "labels": {
        "names": ["toxic", "severe_toxic", "obscene", "threat", "insult", "identity_hate"],
        "text_column": "comment_text",
    },
    "metrics": {"track": ["accuracy", "precision", "recall", "f1", "auc_roc"], "threshold": 0.5},
    "tracking": {"use_wandb": False, "project_name": "content-moderation-phase1", "experiment_name": "colab", "log_model": False},
    "seed": 42,
    "calibration": {"method": "temperature_scaling", "validation_split": 0.2, "n_bins": 10},
    "thresholds": {
        "optimization_metric": "f1",
        "search_method": "grid",
        "search_range": [0.1, 0.9],
        "search_steps": 30,
        "tiers": {
            "auto_remove": {"target_precision": 0.95},
            "human_review": {"target_recall": 0.85},
            "auto_approve": {"target_precision": 0.98},
        },
    },
}

PHASE2_CONFIG = {
    "model": {
        "name": "xlm-roberta-base",
        "max_length": 256,
        "dropout": 0.1,
        "hidden_size": 256,
        "use_language_adapter": False,
    },
    "training": {
        "data_dir": "data",
        "batch_size": 16,
        "learning_rate": 3.0e-5,
        "num_epochs": NUM_EPOCHS,
        "warmup_steps": 100,
        "weight_decay": 0.01,
        "max_grad_norm": 1.0,
        "optimizer": "adamw",
        "scheduler": "linear",
        "device": "cuda",
        "mixed_precision": True,
        "num_workers": 2,
        "pin_memory": True,
        "save_dir": "models",
        "save_strategy": "best",
        "save_total_limit": 2,
        "load_best_at_end": True,
        "eval_strategy": "epoch",
        "eval_steps": 500,
        "logging_steps": 50,
        "early_stopping": True,
        "early_stopping_patience": 2,
        "early_stopping_metric": "f1",
    },
    "languages": {
        "supported": ["english", "hindi", "tamil", "telugu", "kannada", "bengali"],
        "code_mixed": ["hinglish", "tanglish", "tenglish", "kanglish"],
    },
    "tracking": {"use_wandb": False, "project_name": "content-moderation-phase2", "experiment_name": "colab", "log_model": False},
    "seed": 42,
}

phase1_config = copy.deepcopy(PHASE1_CONFIG)
phase2_config = copy.deepcopy(PHASE2_CONFIG)

if PROFILE == "full":
    phase1_config["training"]["use_sample"] = False
    phase1_config["training"]["batch_size"] = 16
    phase2_config["training"]["batch_size"] = 8
    phase1_config["model"]["max_length"] = 512
    phase2_config["model"]["max_length"] = 512

phase1_config_path = "phase1_text_baseline/configs/colab_run.yaml"
phase2_config_path = "phase2_multilingual/configs/colab_run.yaml"

for path in (phase1_config_path, phase2_config_path):
    os.makedirs(os.path.dirname(path), exist_ok=True)

with open(phase1_config_path, "w") as f:
    yaml.dump(phase1_config, f, default_flow_style=False)

with open(phase2_config_path, "w") as f:
    yaml.dump(phase2_config, f, default_flow_style=False)

print("Wrote:", phase1_config_path)
print("Wrote:", phase2_config_path)

## 5. Phase 1 — Download data, train, calibrate

In [ ]:
import time

os.chdir(f"{REPO_DIR}/phase1_text_baseline")
t0 = time.time()

# Patch sklearn metrics bug (float preds vs int labels) on GitHub clone
train_path = "train_classifier.py"
train_src = open(train_path).read()
if "astype(int)" not in train_src.split("precision_recall_fscore_support")[1][:500]:
    train_src = train_src.replace(
        "label_preds = all_preds[:, i]\n        label_labels = all_labels[:, i]",
        "label_preds = all_preds[:, i].astype(int)\n        label_labels = all_labels[:, i].astype(int)",
    )
    train_src = train_src.replace(
        "all_labels.ravel(), all_preds.ravel(), average='binary', zero_division=0",
        "all_labels.ravel().astype(int), all_preds.ravel().astype(int), average='binary', zero_division=0",
    )
    open(train_path, "w").write(train_src)
    print("Patched train_classifier.py for sklearn compatibility")

if PROFILE == "full":
    !python download_data.py --output-dir data --analyze
else:
    if not os.path.exists("data/train.csv"):
        !python download_data.py --output-dir data
    !python download_data.py --output-dir data --create-sample --sample-size {SAMPLE_SIZE}

!python train_classifier.py --config configs/colab_run.yaml
!python calibrate_thresholds.py --config configs/colab_run.yaml

print(f"\nPhase 1 finished in {(time.time() - t0) / 60:.1f} min")
!ls -lh models/
!ls -lh models/calibration/ 2>/dev/null || true

In [ ]:
# Show Phase 1 calibration plots inline
from IPython.display import Image, display
import glob

for path in sorted(glob.glob("models/calibration/*.png")):
    print(path)
    display(Image(filename=path, width=700))

In [ ]:
# Phase 1 metrics summary
import json

cal_path = "models/calibration/calibration_results.json"
if os.path.exists(cal_path):
    with open(cal_path) as f:
        cal = json.load(f)
    print(json.dumps(cal.get("summary", cal), indent=2)[:4000])
else:
    print("calibration_results.json not found")

## 6. Phase 2 — Language ID, data prep, train, evaluate

**If you see `numpy.dtype size changed`:** Runtime → **Restart runtime**, then re-run from cell 1 (Phase 1 models are saved on disk in `/content/` until session ends — if you restart, re-run Phase 1 too).

In [ ]:
# Prepare Phase 2 data inline (bypasses buggy prepare_datasets.py on GitHub clone)
import random
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split

PHASE2_DATA = Path(f"{REPO_DIR}/phase2_multilingual/data")
PHASE2_DATA.mkdir(parents=True, exist_ok=True)

def create_code_mix(size: int, seed: int = 42) -> pd.DataFrame:
    templates = {
        "toxic": [
            ("You are such a {hindi_insult} yaar", 1),
            ("Stop being {hindi_insult}, it's annoying", 1),
            ("Tu bada {english_insult} hai boss", 1),
            ("Kya {english_bad} bakwas hai ye", 1),
        ],
        "neutral": [
            ("Aaj main bahut {hindi_good} feel kar raha hoon", 0),
            ("This movie was {hindi_adj} good yaar", 0),
            ("Main kal {english_place} ja raha hoon", 0),
            ("Yeh {english_thing} bohot accha hai", 0),
        ],
    }
    hindi_words = {
        "insult": ["bewakoof", "pagal", "badtameez"],
        "bad": ["ganda", "bura", "bekaar"],
        "good": ["khush", "accha", "mast"],
        "adj": ["zabardast", "kamaal", "shandar"],
    }
    english_words = {
        "insult": ["idiot", "fool", "jerk"],
        "bad": ["terrible", "awful", "horrible"],
        "place": ["school", "office", "market"],
        "thing": ["phone", "laptop", "car"],
    }
    rng = random.Random(seed)
    rows = []
    for _ in range(size):
        category = rng.choices(["toxic", "neutral"], weights=[0.3, 0.7], k=1)[0]
        template, label = rng.choice(templates[category])
        text = template
        for key, bank in [
            ("hindi_insult", hindi_words["insult"]),
            ("hindi_bad", hindi_words["bad"]),
            ("hindi_good", hindi_words["good"]),
            ("hindi_adj", hindi_words["adj"]),
            ("english_insult", english_words["insult"]),
            ("english_bad", english_words["bad"]),
            ("english_place", english_words["place"]),
            ("english_thing", english_words["thing"]),
        ]:
            if f"{{{key}}}" in text:
                text = text.replace(f"{{{key}}}", rng.choice(bank))
        rows.append({"text": text, "label": label, "language": "hinglish", "is_code_mixed": True})
    return pd.DataFrame(rows)

# English from Phase 1
phase1_csv = Path(f"{REPO_DIR}/phase1_text_baseline/data/train_sample.csv")
if not phase1_csv.exists():
    phase1_csv = Path(f"{REPO_DIR}/phase1_text_baseline/data/train.csv")

df_en = pd.read_csv(phase1_csv)
if len(df_en) > PHASE2_SAMPLE:
    df_en = df_en.sample(PHASE2_SAMPLE, random_state=42)

label_cols = ["toxic", "severe_toxic", "obscene", "threat", "insult", "identity_hate"]
df_en["label"] = (df_en[label_cols].sum(axis=1) > 0).astype(int)
df_en = df_en.rename(columns={"comment_text": "text"})[["text", "label"]]
df_en["language"] = "english"
df_en["is_code_mixed"] = False

df_mix = create_code_mix(PHASE2_SAMPLE)
combined = pd.concat([df_en, df_mix], ignore_index=True).sample(frac=1, random_state=42)

train_df, val_df = train_test_split(
    combined,
    test_size=0.1,
    random_state=42,
    stratify=combined["language"] + "_" + combined["label"].astype(str),
)

train_df.to_csv(PHASE2_DATA / "multilingual_train_split.csv", index=False)
val_df.to_csv(PHASE2_DATA / "multilingual_val_split.csv", index=False)
combined.to_csv(PHASE2_DATA / "multilingual_train.csv", index=False)

print(f"Phase 2 data ready: train={len(train_df)}, val={len(val_df)}")
print(combined["language"].value_counts())

In [ ]:
import time

os.chdir(f"{REPO_DIR}/phase2_multilingual")
t0 = time.time()
os.makedirs("models", exist_ok=True)

if not os.path.exists("models/lid.176.bin"):
    !wget -q -O models/lid.176.bin https://dl.fbaipublicfiles.com/fasttext/supervised-models/lid.176.bin

# Language routing smoke test (FastText warnings are OK — fallback handles them)
import importlib
import language_identifier
importlib.reload(language_identifier)
from language_identifier import LanguageIdentifier

idf = LanguageIdentifier(fasttext_model_path="models/lid.176.bin")
for t in ["This is English", "Aaj main bahut khush hoon yaar", "यह हिंदी है"]:
    r = idf.identify(t)
    print(f"{t[:35]:35s} -> {r['language']:10s} ({r['confidence']:.2f})")

# Data already prepared in previous cell — train directly
assert os.path.exists("data/multilingual_train_split.csv"), "Run the data prep cell first"
!python train_multilingual.py --config configs/colab_run.yaml

print(f"\nPhase 2 training finished in {(time.time() - t0) / 60:.1f} min")

In [ ]:
# Phase 2 evaluation
import json
import pandas as pd

if os.path.exists("models/val_predictions.npy") and os.path.exists("models/val_labels.npy"):
    val_df = pd.read_csv("data/multilingual_val_split.csv")
    with open("models/val_languages.json", "w") as f:
        json.dump(val_df["language"].tolist(), f)

    !python evaluate_multilingual.py \
        --predictions models/val_predictions.npy \
        --labels models/val_labels.npy \
        --languages models/val_languages.json \
        --output-dir evaluation

    if os.path.exists("evaluation/evaluation_report.md"):
        print(open("evaluation/evaluation_report.md").read())
else:
    print("Validation predictions not found — skipping evaluation.")

In [ ]:
# Show per-language performance plot
plot_path = "evaluation/per_language_performance.png"
if os.path.exists(plot_path):
    display(Image(filename=plot_path, width=700))

## 7. Unified inference demo (Phase 1 + Phase 2)

In [ ]:
import subprocess
import sys

PHASE1_MODEL = "../phase1_text_baseline/models/best_model.pt"
PHASE1_CALIB = "../phase1_text_baseline/models/calibration/calibration_results.json"
PHASE2_MODEL = "models/best_model.pt"
LANG_MODEL = "models/lid.176.bin"

test_texts = [
    "You are an idiot and I hate you",
    "Great article, thanks for sharing!",
    "Aaj tu bahut pagal hai yaar",
    "Kya bakwas hai ye, stop this nonsense",
]

for text in test_texts:
    print("\n" + "=" * 60)
    subprocess.run(
        [
            sys.executable,
            "unified_inference.py",
            "--phase1-model", PHASE1_MODEL,
            "--phase2-model", PHASE2_MODEL,
            "--language-model", LANG_MODEL,
            "--calibration", PHASE1_CALIB,
            "--text", text,
        ],
        check=False,
    )

## 8. Download artifacts to your Mac

In [ ]:
import shutil
from google.colab import files

os.chdir(REPO_DIR)

ARTIFACT_DIR = "/content/trained_artifacts"
if os.path.exists(ARTIFACT_DIR):
    shutil.rmtree(ARTIFACT_DIR)
os.makedirs(ARTIFACT_DIR, exist_ok=True)

# Copy key outputs (skip huge raw CSV)
paths_to_copy = [
    "phase1_text_baseline/models/best_model.pt",
    "phase1_text_baseline/models/val_predictions.npy",
    "phase1_text_baseline/models/val_labels.npy",
    "phase1_text_baseline/models/calibration",
    "phase2_multilingual/models/best_model.pt",
    "phase2_multilingual/models/lid.176.bin",
    "phase2_multilingual/models/val_predictions.npy",
    "phase2_multilingual/models/val_labels.npy",
    "phase2_multilingual/evaluation",
    "phase1_text_baseline/configs/colab_run.yaml",
    "phase2_multilingual/configs/colab_run.yaml",
]

for rel in paths_to_copy:
    src = os.path.join(REPO_DIR, rel)
    dst = os.path.join(ARTIFACT_DIR, rel)
    if os.path.exists(src):
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        if os.path.isdir(src):
            shutil.copytree(src, dst)
        else:
            shutil.copy2(src, dst)
        print(f"copied {rel}")
    else:
        print(f"skip (missing): {rel}")

zip_path = "/content/content_moderation_trained.zip"
shutil.make_archive("/content/content_moderation_trained", "zip", ARTIFACT_DIR)

size_mb = os.path.getsize(zip_path) / (1024 * 1024)
print(f"\nZip ready: {zip_path} ({size_mb:.1f} MB)")
files.download(zip_path)

## 9. Restore artifacts locally (run on your Mac after download)

```bash
cd content-moderation-system
unzip ~/Downloads/content_moderation_trained.zip -d /tmp/artifacts
cp -r /tmp/artifacts/phase1_text_baseline/models/* phase1_text_baseline/models/
cp -r /tmp/artifacts/phase2_multilingual/models/* phase2_multilingual/models/
cp -r /tmp/artifacts/phase2_multilingual/evaluation phase2_multilingual/

# Demo inference locally (CPU is fine)
cd phase2_multilingual
python unified_inference.py \
  --phase1-model ../phase1_text_baseline/models/best_model.pt \
  --phase2-model models/best_model.pt \
  --language-model models/lid.176.bin \
  --calibration ../phase1_text_baseline/models/calibration/calibration_results.json \
  --text "Aaj tu bahut pagal hai yaar"
```